# Module 09 — The Data Model: Dunder Methods

## Exercise 09.2 — A Matrix that behaves like a built-in type

Implement enough of the data model that Matrix works with Python's syntax
rather than beside it. The tests are the specification.
Run:  python ex02_matrix.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 1. `__repr__` and `__str__`

Implement `__repr__` on every class you write. It costs one line and it pays
back in every debugging session.

In [ ]:
class Point:
    def __init__(self, x: float, y: float) -> None:
        self.x, self.y = x, y

    def __repr__(self) -> str:
        return f"Point(x={self.x!r}, y={self.y!r})"    # for DEVELOPERS

    def __str__(self) -> str:
        return f"({self.x}, {self.y})"                  # for USERS

| | `__repr__` | `__str__` |
|---|---|---|
| Audience | Developers | End users |
| Goal | Unambiguous | Readable |
| Called by | REPL, `repr()`, containers, debuggers, logging | `print()`, `str()`, f-strings |
| Default | `<Point object at 0x7f...>` | Falls back to `__repr__` |
| Ideal | Valid Python that reconstructs the object | Whatever reads best |

Define `__repr__` always; define `__str__` only when the user-facing form
genuinely differs.

**The rule that matters:** a container's `str()` uses its elements' `repr()`.

```text
>>> print([Point(1, 2)])
[Point(x=1, y=2)]           # __repr__, not __str__
```


So a class with only `__str__` still prints as `<object at 0x...>` inside a
list, which is exactly when you most need to see it. And use `!r` inside your
repr: `f"{self.name!r}"` shows `'Ada'` rather than `Ada`, which distinguishes an
empty string from a missing value.

---

## Concept 4. The container protocols

In [ ]:
class Deck:
    def __init__(self, cards: list[str]) -> None:
        self._cards = list(cards)

    def __len__(self) -> int:                 # len(deck)
        return len(self._cards)

    def __getitem__(self, index):             # deck[0], deck[1:3]
        return self._cards[index]             # slices work for free

    def __setitem__(self, index, value) -> None:
        self._cards[index] = value

    def __delitem__(self, index) -> None:
        del self._cards[index]

    def __contains__(self, card: str) -> bool:  # "AS" in deck
        return card in self._cards

    def __iter__(self):                        # for card in deck
        return iter(self._cards)

    def __reversed__(self):                    # reversed(deck)
        return reversed(self._cards)

**`__getitem__` alone gives you a great deal.** Without `__iter__`, Python falls
back to calling `__getitem__` with 0, 1, 2, ... until `IndexError`. So iteration,
`in`, `list()`, and unpacking all work from `__getitem__` alone. This is the old
protocol, kept for compatibility — implement `__iter__` explicitly anyway,
because the fallback only works for integer-indexed sequences and produces
confusing errors when it does not apply.

**Handling slices:** `__getitem__` receives a `slice` object for `deck[1:3]`.
Delegating to a list (as above) handles it automatically. If you build the result
yourself, return **your own type** for a slice and a single element for an int:

In [ ]:
def __getitem__(self, index):
    if isinstance(index, slice):
        return type(self)(self._cards[index])     # type(self), not Deck --
    return self._cards[index]                     # subclasses get their type

---

## Concept 7. Operators

In [ ]:
class Vector:
    def __init__(self, x: float, y: float) -> None:
        self.x, self.y = x, y

    def __add__(self, other: "Vector") -> "Vector":
        if not isinstance(other, Vector):
            return NotImplemented
        return Vector(self.x + other.x, self.y + other.y)

    def __mul__(self, scalar: float) -> "Vector":
        if not isinstance(scalar, (int, float)):
            return NotImplemented
        return Vector(self.x * scalar, self.y * scalar)

    __rmul__ = __mul__            # makes 3 * v work as well as v * 3

    def __neg__(self) -> "Vector":
        return Vector(-self.x, -self.y)

    def __abs__(self) -> float:
        return (self.x**2 + self.y**2) ** 0.5

**How Python resolves `a + b`:**

1. Try `type(a).__add__(a, b)`. If it returns `NotImplemented`, continue.
2. Try `type(b).__radd__(b, a)`. If that also returns `NotImplemented`:
3. `TypeError: unsupported operand type(s)`.

(With one refinement: if `type(b)` is a *subclass* of `type(a)`, the reflected
method is tried first, so a subclass can override its parent's behaviour.)

This is why `NotImplemented` matters. Returning it is how you say "not my
problem" and let the other operand try. Note the trap: `NotImplemented` is
**truthy**, so accidentally returning it from `__eq__` and using the result in an
`if` gives you a silent wrong answer plus a `DeprecationWarning`.

**In-place operators** (`__iadd__` etc.) should mutate and `return self` — for a
mutable type. For an immutable one, omit them and Python falls back to
`__add__` plus rebinding. This is exactly Module 02's list-versus-tuple `+=`
distinction, now from the implementer's side.

Only overload operators where the meaning is obvious. `Vector + Vector` is
clear. `User + User` is not, and a `merge()` method would be better.

---

## Concept 9. The whole map

| Group | Methods |
|---|---|
| Representation | `__repr__` `__str__` `__format__` `__bytes__` |
| Comparison | `__eq__` `__ne__` `__lt__` `__le__` `__gt__` `__ge__` `__hash__` |
| Container | `__len__` `__getitem__` `__setitem__` `__delitem__` `__contains__` `__reversed__` |
| Iteration | `__iter__` `__next__` `__aiter__` `__anext__` |
| Numeric | `__add__` `__sub__` `__mul__` `__truediv__` `__floordiv__` `__mod__` `__pow__` `__neg__` `__abs__` `__round__` and the `__r*__` / `__i*__` variants |
| Conversion | `__bool__` `__int__` `__float__` `__index__` `__complex__` |
| Context | `__enter__` `__exit__` `__aenter__` `__aexit__` |
| Callable | `__call__` |
| Attributes | `__getattr__` `__getattribute__` `__setattr__` `__delattr__` `__dir__` |
| Descriptors | `__get__` `__set__` `__delete__` `__set_name__` |
| Class machinery | `__init__` `__new__` `__init_subclass__` `__class_getitem__` `__slots__` |
| Copying | `__copy__` `__deepcopy__` `__reduce__` |
| Pattern matching | `__match_args__` |

You do not need to memorise this. You need to know it exists, so that when you
want your type to work with some piece of syntax, you look up which method
provides it.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: `__repr__` and `__str__`
- Section 2: `__eq__` and `__hash__` are a pair
- Section 3: Ordering
- Section 4: The container protocols
- Section 5: Iteration
- Section 6: Context managers
- Section 7: Operators
- Section 8: `__call__`, `__bool__`, `__format__`
- Section 9: The whole map

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

from collections.abc import Iterator, Sequence

---

## `Matrix`

A 2-D matrix of floats.

In [ ]:
class Matrix:
    """A 2-D matrix of floats.

    TODO 1  __init__(rows) where rows is a sequence of equal-length sequences.
            Validate: non-empty, rectangular, numeric. Store immutably enough
            that a caller cannot reach in and change it (Module 08).

    TODO 2  __repr__ -- unambiguous, ideally reconstructing.
            __str__  -- an aligned grid a human can read.

    TODO 3  __eq__ and __hash__. Decide whether Matrix is hashable AT ALL and
            justify it in a comment. (Think about what you chose in TODO 1.)

    TODO 4  __len__  -- number of rows. Say why that is the right answer rather
            than the number of cells, in one comment line.

    TODO 5  __getitem__ supporting THREE index forms:
              m[1]        -> a row (as a tuple)
              m[1, 2]     -> a single cell
              m[0:2]      -> a Matrix of those rows -- note: your OWN type
            Use type(self)(...) for the slice case, not Matrix(...), and say why.

    TODO 6  __iter__ yielding rows. Two consecutive for loops must both work.

    TODO 7  __contains__ -- is a VALUE present anywhere in the matrix?
            Note this differs from the default derived from __iter__, which
            would test whether a ROW is present. Decide which is less
            surprising and write down the reasoning.

    TODO 8  __add__ (elementwise, same shape), __mul__ (by a scalar OR by
            another Matrix as matrix multiplication), __rmul__, __neg__.
            Return NotImplemented for types you do not handle, and raise
            ValueError for shape mismatches. Explain the difference between
            those two responses in a comment -- it is not arbitrary.

    TODO 9  __bool__ -- what should an all-zero matrix be? Justify.

    TODO 10 __format__ supporting f"{m:.2f}" to control cell formatting.

    TODO 11 transpose() and a `shape` property.
    """

---

## `verify`

_verify_

In [ ]:
def verify() -> None:
    m = Matrix([[1, 2, 3], [4, 5, 6]])

    assert m.shape == (2, 3)
    assert len(m) == 2
    assert m[0] == (1, 2, 3)
    assert m[1, 2] == 6
    assert m[0:1] == Matrix([[1, 2, 3]])
    assert isinstance(m[0:1], Matrix)

    assert list(m) == [(1, 2, 3), (4, 5, 6)]
    assert list(m) == [(1, 2, 3), (4, 5, 6)], "two loops must both work"

    assert 5 in m
    assert 99 not in m

    assert m + m == Matrix([[2, 4, 6], [8, 10, 12]])
    assert m * 2 == Matrix([[2, 4, 6], [8, 10, 12]])
    assert 2 * m == Matrix([[2, 4, 6], [8, 10, 12]])
    assert -m == Matrix([[-1, -2, -3], [-4, -5, -6]])

    a = Matrix([[1, 2], [3, 4]])
    b = Matrix([[5, 6], [7, 8]])
    assert a * b == Matrix([[19, 22], [43, 50]])

    assert m.transpose() == Matrix([[1, 4], [2, 5], [3, 6]])

    try:
        m + Matrix([[1]])
    except ValueError:
        pass
    else:
        raise AssertionError("shape mismatch must raise ValueError")

    assert (m + "not a matrix") is NotImplemented or True   # see the comment
    try:
        m + "not a matrix"      # type: ignore[operator]
    except TypeError:
        pass
    else:
        raise AssertionError("adding a str must raise TypeError")

    assert "Matrix" in repr(m)
    assert "\n" in str(m)
    assert f"{m:.2f}".count(".") >= 6

    assert bool(Matrix([[0, 0], [0, 0]])) is False or True   # your call
    print("all matrix checks passed\n")
    print(m)
    print()
    print(f"{m * 1.5:.2f}")

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    verify()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.